# 04. Model Evaluation & Performance Analysis — ResumeAI
Evaluates the final selected model on held-out test data and plots the confusion matrix.

In [ ]:
import sys
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Load Model & Vectorizer

In [ ]:
model = joblib.load('../models/classifier/best_classifier.joblib')
vectorizer = joblib.load('../models/vectorizer/tfidf_vectorizer.joblib')

with open('../models/model_metadata.json', 'r') as f:
    meta = json.load(f)

print(f'Loaded Best Model: {meta.get("best_model_name")}')
print(f'Test Accuracy Recorded: {meta.get("test_metrics", {}).get("accuracy", 0) * 100:.2f}%')

## 2. Test Set Evaluation

In [ ]:
df = pd.read_csv('../data/processed/cleaned_resumes.csv')
X = df['Cleaned_Resume'].astype(str).values
y = df['Category'].astype(str).values

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_test_vec = vectorizer.transform(X_test)
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred, zero_division=0))

## 3. Confusion Matrix Plot

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=meta['classes'])
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=meta['classes'], yticklabels=meta['classes'])
plt.title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Domain')
plt.ylabel('Actual Domain')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()